# Week 2 — Data Handling with NumPy & Pandas
**AI/ML Internship — Code-IT**
**Author:** Sankalpa Parajuli

This notebook covers the Week 2 deliverable: loading a raw dataset, cleaning it,
and producing an analysis-ready version, with comments explaining each step.
The dataset (`raw_employee_data.csv`) is a small synthetic employee dataset with
typical real-world issues: missing values, duplicate rows, and inconsistent text formatting.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

## 1. Load the Raw Dataset

In [2]:
raw_df = pd.read_csv("raw_employee_data.csv")
print("Shape:", raw_df.shape)
raw_df.head(10)

Shape: (21, 7)


,employee_id,name,department,city,age,salary,years_experience
0,101,Aarav Shrestha,Marketing,Pokhara,52.0,79127.0,8.0
1,102,Priya Gurung,IT,Dharan,35.0,37994.0,13.0
2,103,bishal rai,Finance,NaN,26.0,77318.0,13.0
3,104,Sita Tamang,Marketing,Itahari,NaN,25478.0,9.0
4,105,Kiran Thapa,Sales,Pokhara,42.0,76794.0,17.0
5,106,NISHA ADHIKARI,NaN,Kathmandu,44.0,76150.0,NaN
6,107,Rohit Karki,Marketing,Biratnagar,24.0,75723.0,1.0
7,108,Anjali Basnet,sales,Kathmandu,31.0,68215.0,15.0
8,109,Suman Magar,NaN,Kathmandu,47.0,55651.0,11.0
9,110,Deepa Poudel,HR,Pokhara,49.0,70835.0,12.0


## 2. Initial Inspection
Before cleaning anything, check data types, missing values, and obvious issues.

In [3]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   employee_id       21 non-null     int64  
 1   name              21 non-null     str    
 2   department        19 non-null     str    
 3   city              19 non-null     str    
 4   age               19 non-null     float64
 5   salary            20 non-null     float64
 6   years_experience  20 non-null     float64
dtypes: float64(3), int64(1), str(3)
memory usage: 1.3 KB


In [4]:
# Count missing values per column
raw_df.isnull().sum()

employee_id         0
name                0
department          2
city                2
age                 2
salary              1
years_experience    1
dtype: int64

In [5]:
# Check for duplicate rows
print("Duplicate rows:", raw_df.duplicated().sum())

Duplicate rows: 1


## 3. Cleaning Steps

We'll clean the dataset in a clear sequence:
1. Remove duplicate rows
2. Fix inconsistent text (extra whitespace, mixed case) in `name` and `department`
3. Handle missing values in `department`, `age`, `salary`, and `years_experience`
4. Convert columns to appropriate data types

In [6]:
df = raw_df.copy()

# 3.1 Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate row(s).")

Removed 1 duplicate row(s).


In [7]:
# 3.2 Clean up text columns: strip whitespace and standardise casing
df["name"] = df["name"].str.strip().str.title()
df["department"] = df["department"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.title()

df[["name", "department", "city"]].head(10)

,name,department,city
0,Aarav Shrestha,Marketing,Pokhara
1,Priya Gurung,It,Dharan
2,Bishal Rai,Finance,NaN
3,Sita Tamang,Marketing,Itahari
4,Kiran Thapa,Sales,Pokhara
5,Nisha Adhikari,NaN,Kathmandu
6,Rohit Karki,Marketing,Biratnagar
7,Anjali Basnet,Sales,Kathmandu
8,Suman Magar,NaN,Kathmandu
9,Deepa Poudel,Hr,Pokhara


In [8]:
# 3.3 Handle missing values
# - department/city: fill with "Unknown" since we can't guess these
# - age/years_experience: fill with the column median (robust to outliers)
# - salary: fill with the median salary within the same department

df["department"] = df["department"].fillna("Unknown")
df["city"] = df["city"].fillna("Unknown")

df["age"] = df["age"].fillna(df["age"].median())
df["years_experience"] = df["years_experience"].fillna(df["years_experience"].median())

df["salary"] = df.groupby("department")["salary"].transform(
    lambda x: x.fillna(x.median())
)

df.isnull().sum()

employee_id         0
name                0
department          0
city                0
age                 0
salary              0
years_experience    0
dtype: int64

In [9]:
# 3.4 Fix data types now that missing values are handled
df["age"] = df["age"].astype(int)
df["years_experience"] = df["years_experience"].astype(int)
df["salary"] = df["salary"].round(2)

df.dtypes

employee_id           int64
name                    str
department              str
city                    str
age                   int64
salary              float64
years_experience      int64
dtype: object

## 4. Filtering, Grouping, and Pivoting
A few practice operations required for Week 2.

In [10]:
# Filter: employees with more than 10 years of experience
experienced = df[df["years_experience"] > 10]
experienced[["name", "department", "years_experience"]]

,name,department,years_experience
1,Priya Gurung,It,13
2,Bishal Rai,Finance,13
4,Kiran Thapa,Sales,17
5,Nisha Adhikari,Unknown,11
7,Anjali Basnet,Sales,15
8,Suman Magar,Unknown,11
9,Deepa Poudel,Hr,12
10,Manish Lama,It,11
11,Sarita Bhandari,Finance,11
13,Puja Shah,Finance,11


In [11]:
# Groupby: average salary per department
avg_salary_by_dept = df.groupby("department")["salary"].mean().sort_values(ascending=False)
avg_salary_by_dept

department
Unknown      65900.500000
Hr           64349.500000
Finance      62212.500000
Sales        59698.333333
Marketing    54777.800000
It           47658.000000
Name: salary, dtype: float64

In [12]:
# Pivot table: average age and salary by department and city
pivot = pd.pivot_table(
    df,
    values=["age", "salary"],
    index="department",
    columns="city",
    aggfunc="mean"
)
pivot

age                                              salary  \
city       Biratnagar Dharan Itahari Kathmandu Pokhara Unknown Biratnagar   
department                                                                  
Finance           NaN   38.0     NaN       NaN     NaN    37.0        NaN   
Hr               51.0    NaN     NaN      29.0    49.0     NaN    57864.0   
It                NaN   35.0     NaN      30.0    35.0     NaN        NaN   
Marketing        24.0   49.0    42.0       NaN    38.5     NaN    75723.0   
Sales            44.0    NaN     NaN      31.0    42.0     NaN    34086.0   
Unknown           NaN    NaN     NaN      45.5     NaN     NaN        NaN   

                                                          
city         Dharan  Itahari Kathmandu  Pokhara  Unknown  
department                                                
Finance     47892.5      NaN       NaN      NaN  76532.5  
Hr              NaN      NaN   64349.5  70835.0      NaN  
It          37994.0      NaN   61968.0  43012.0      NaN  
Marketing   61117.0  25478.0       NaN  55785.5      NaN  
Sales           NaN      NaN   68215.0  76794.0      NaN  
Unknown         NaN      NaN   65900.5      NaN      NaN

## 5. Save the Clean Dataset
The cleaned, analysis-ready dataset is written out as `clean_employee_data.csv`.

In [13]:
df.to_csv("clean_employee_data.csv", index=False)
df.head(10)

,employee_id,name,department,city,age,salary,years_experience
0,101,Aarav Shrestha,Marketing,Pokhara,52,79127.0,8
1,102,Priya Gurung,It,Dharan,35,37994.0,13
2,103,Bishal Rai,Finance,Unknown,26,77318.0,13
3,104,Sita Tamang,Marketing,Itahari,42,25478.0,9
4,105,Kiran Thapa,Sales,Pokhara,42,76794.0,17
5,106,Nisha Adhikari,Unknown,Kathmandu,44,76150.0,11
6,107,Rohit Karki,Marketing,Biratnagar,24,75723.0,1
7,108,Anjali Basnet,Sales,Kathmandu,31,68215.0,15
8,109,Suman Magar,Unknown,Kathmandu,47,55651.0,11
9,110,Deepa Poudel,Hr,Pokhara,49,70835.0,12


## 6. Summary

- Started with 21 rows (including 1 duplicate) and several missing/inconsistent values.
- Removed the duplicate row and standardised text formatting in `name`, `department`, and `city`.
- Filled missing numeric values using median (and department-wise median for salary) rather than dropping rows, to keep the dataset size intact.
- Final dataset has no missing values and consistent formatting, ready for further analysis (Week 3's EDA).